# YouTube Auto-Dub — dub + lip-sync on a free Colab GPU

Runtime → **Change runtime type → GPU (T4)**. This dubs a YouTube video into
another language *in the original voice* and (optionally) re-renders the mouth to
match, using Wav2Lip. Everything is open-source and free.

## 1. Check the GPU

In [1]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-61a89c20-e8d1-c77b-b1f7-ebfdd1d3bbb7)


## 2. Install ffmpeg + ytdub
The `[xtts,nllb]` extras pull the cloning TTS and the neural translator. On the
Colab GPU runtime torch already has CUDA, so torchcodec loads fine here.

In [5]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg python3-venv > /dev/null
!ffmpeg -version | head -n 1

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


## 3. Pick a video and dub it (no lip-sync yet)
First run downloads the models (~4 GB). Output lands in `data/output/`.

In [ ]:
import os, sys, shutil, subprocess

os.chdir("/content")
VENV = "/content/ytdub-env"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["VIRTUAL_ENV"] = VENV
os.environ["PATH"] = f"{VENV}/bin:" + os.environ.get("PATH", "")
os.makedirs("/content/hf_cache", exist_ok=True)
os.makedirs("/content/data/output", exist_ok=True)

print("Python di Colab (NON lo usiamo):", sys.version)

!pip -q install -U uv
!uv python install 3.12
!uv venv /content/ytdub-env --python 3.12 --seed

!uv pip install --python /content/ytdub-env/bin/python \
  torch torchaudio --index-url https://download.pytorch.org/whl/cu124

!uv pip install --python /content/ytdub-env/bin/python -U yt-dlp
!uv pip install --python /content/ytdub-env/bin/python \
  "ytdub[chatterbox,nllb] @ git+https://github.com/mazzasaverio/youtube-auto-dub.git"

os.environ["PATH"] = "/content/ytdub-env/bin:" + os.environ["PATH"]

print("\npython  ->", shutil.which("python"))
print("ytdub   ->", shutil.which("ytdub"))
!/content/ytdub-env/bin/python --version
!/content/ytdub-env/bin/ytdub info

print("\nOK. NON riavviare. Vai alla CELLA 4.")

Python di Colab (NON lo usiamo): 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 91.7 MB/s eta 0:00:00
Installed Python 3.12.14 in 2.21s
 + cpython-3.12.14-linux-x86_64-gnu (python3.12)
Using CPython 3.12.14
Creating virtual environment with seed packages at: ytdub-env
 + pip==26.2.1
Activate with: source ytdub-env/bin/activate
Using Python 3.12.14 environment at: ytdub-env
Resolved 25 packages in 359ms
⠸ Preparing packages... (16/25)                                                 

## 4. Set up Wav2Lip (its own venv — its deps conflict with coqui-tts)
We create a separate virtualenv for Wav2Lip and download its checkpoint. If the
checkpoint URL 404s, grab `wav2lip_gan.pth` from the Wav2Lip README and drop it in
`Wav2Lip/checkpoints/`.

In [ ]:
import os, glob, shutil, subprocess, sys
from pathlib import Path

os.chdir("/content")
VENV = "/content/ytdub-env"
os.environ["VIRTUAL_ENV"] = VENV
os.environ["PATH"] = f"{VENV}/bin:" + os.environ.get("PATH", "")
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["COQUI_TOS_AGREED"] = "1"
os.makedirs("/content/hf_cache", exist_ok=True)
os.makedirs("/content/data/output", exist_ok=True)

ytdub_bin = shutil.which("ytdub")
if not ytdub_bin or not ytdub_bin.startswith(VENV):
    raise SystemExit(
        "ytdub non è nel venv. Riesegui la CELLA 3 e NON riavviare la sessione.\n"
        f"which ytdub = {ytdub_bin}"
    )

# ============== MODIFICA SOLO QUI ==============
SOURCE = "/content/video.mp4"     # file caricato  OPPURE  URL YouTube
TARGET = "en"                     # it, en, es, fr, de, pt, ...
ASR_MODEL = "small"               # tiny | base | small | medium
TRANSLATOR = "nllb"               # se OOM: "argos"
TTS = "chatterbox"                # se crasha: "xtts"
BURN_SUBS = False                 # True solo se vuoi caption bruciate nel video
MAX_HEIGHT = 720
COOKIES = "/content/cookies.txt"  # solo se SOURCE è un URL e YouTube blocca
# ===============================================

def run(cmd, check=True):
    print("\n>>", " ".join(cmd) if isinstance(cmd, list) else cmd, flush=True)
    r = subprocess.run(cmd if isinstance(cmd, list) else cmd, shell=not isinstance(cmd, list))
    if check and r.returncode != 0:
        raise RuntimeError(f"comando fallito ({r.returncode})")
    return r

print("Colab Python :", sys.version.split()[0], "(ignorato)")
print("venv Python  :", subprocess.check_output([f"{VENV}/bin/python", "--version"], text=True).strip())
print("ytdub        :", ytdub_bin)
print("SOURCE       :", SOURCE)

## 5. Dub **with lip-sync**
Point ytdub at the Wav2Lip venv and re-run with `--lipsync`. Wav2Lip re-renders the
mouth to match the dubbed audio; the result is remuxed as a share-ready MP4.

In [ ]:
from google.colab import files

def is_url(s):
    return str(s).startswith("http://") or str(s).startswith("https://")

video_path = None

if is_url(SOURCE):
    out_tmpl = "/content/source.%(ext)s"
    for old in glob.glob("/content/source.*"):
        os.remove(old)

    cmd = [
        "yt-dlp", "--no-playlist", "--merge-output-format", "mp4",
        "-S", f"res:{MAX_HEIGHT}",
        "-f", f"bv*[height<={MAX_HEIGHT}]+ba/b[height<={MAX_HEIGHT}]/b",
        "--extractor-args", "youtube:player_client=android,web",
        "-o", out_tmpl, SOURCE,
    ]
    if os.path.isfile(COOKIES):
        cmd[1:1] = ["--cookies", COOKIES]
        print("uso cookies:", COOKIES)
    else:
        print("niente cookies.txt — su Colab YouTube spesso fallisce")

    r = run(cmd, check=False)
    found = glob.glob("/content/source.*")
    found = [p for p in found if p.lower().endswith((".mp4", ".mkv", ".webm", ".mov"))]
    if r.returncode != 0 or not found:
        print("\n*** YouTube bloccato (bot-check). Fai UNA di queste:")
        print("  A) carica il video in /content/ e metti SOURCE = '/content/nome.mp4'")
        print("  B) sul PC: estensione 'Get cookies.txt LOCALLY' su youtube.com loggato")
        print("     Upload cookies.txt → /content/cookies.txt e rilancia QUESTA cella")
        print("\nCarica ora un MP4 da questo popup.")
        up = files.upload()
        if not up:
            raise SystemExit("nessun file caricato")
        video_path = "/content/" + list(up.keys())[0]
    else:
        video_path = found[0]
else:
    if not os.path.isfile(SOURCE):
        print(f"{SOURCE} non c'è. Carica il video nel popup (finisce in /content/).")
        up = files.upload()
        if not up:
            raise SystemExit("nessun file caricato")
        video_path = "/content/" + list(up.keys())[0]
    else:
        video_path = SOURCE

video_path = os.path.abspath(video_path)
print("VIDEO PRONTO:", video_path, "size", os.path.getsize(video_path) // (1024*1024), "MB")

## 6. Preview / download the result

In [ ]:
cmd = [
    "ytdub", "dub", video_path,
    "--target", TARGET,
    "--asr-model", ASR_MODEL,
    "--translator", TRANSLATOR,
    "--tts", TTS,
    "--reencode",
]
if BURN_SUBS:
    cmd.append("--subtitles")

run(cmd)

outs = sorted(
    glob.glob("/content/data/output/*.mp4") + glob.glob("data/output/*.mp4"),
    key=os.path.getmtime,
)
if not outs:
    raise SystemExit("nessun mp4 in data/output — guarda l'errore sopra")
dubbed = os.path.abspath(outs[-1])
print("DUB OK:", dubbed)

7

In [ ]:
from IPython.display import Video, display
from google.colab import files

print(dubbed)
display(Video(dubbed, embed=True, width=360))
files.download(dubbed)